In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.morphology import opening, closing, remove_small_objects
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi

def segment_wood(image_path, target_size=(512, 512)):
    image = imread(image_path)
    
    if image.ndim == 3:
        image_gray = rgb2gray(image)
    else:
        image_gray = image.astype(float) / 255.0
    
    # Resize
    image_small = resize(image, 
                         (target_size[0], target_size[1], image.shape[2]) 
                         if image.ndim == 3 else target_size, 
                         anti_aliasing=True)
    image_gray = resize(image_gray, target_size, anti_aliasing=True)
    
    # --- Step 1: Vertical grain suppression ---
    vertical_kernel = np.ones((21, 1))
    grain_estimate = opening(image_gray, vertical_kernel)
    grain_estimate = closing(grain_estimate, vertical_kernel)
    residual = np.abs(image_gray - grain_estimate)
    
    # --- Step 2: Otsu on residual ---
    thresh = threshold_otsu(residual)
    binary = residual > thresh
    
    # --- Step 3: Clean up with morphology ---
    # Fill small holes
    binary = ndi.binary_fill_holes(binary)
    # Remove small spurious objects
    binary = remove_small_objects(binary, min_size=100)
    
    return image_small, image_gray, residual, binary

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    
    valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
    image_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(valid_extensions)
    ]
    
    print(f"Found {len(image_files)} images")
    
    for i, filename in enumerate(image_files):
        print(f"Processing {filename} ({i+1}/{len(image_files)})...", end=' ')
        
        image_path = os.path.join(input_folder, filename)
        
        try:
            image, image_gray, residual, binary = segment_wood(image_path)
            
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            
            axes[0].imshow(image_gray, cmap='gray')
            axes[0].set_title('Grayscale')
            axes[0].axis('off')
            
            axes[1].imshow(residual, cmap='gray')
            axes[1].set_title('After Grain Suppression')
            axes[1].axis('off')
            
            axes[2].imshow(binary, cmap='gray')
            axes[2].set_title('Otsu Threshold')
            axes[2].axis('off')
            
            axes[3].imshow(image_gray, cmap='gray')
            axes[3].contour(binary, [0.5], colors='r', linewidths=1.5)
            axes[3].set_title('Defects Overlaid')
            axes[3].axis('off')
            
            plt.suptitle(filename, fontsize=12)
            plt.tight_layout()
            
            output_name = os.path.splitext(filename)[0] + '_segmented.png'
            output_path = os.path.join(output_folder, output_name)
            plt.savefig(output_path, dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"done -> {output_name}")
            
        except Exception as e:
            print(f"ERROR: {e}")
            plt.close()

input_folder  = 'wood'
output_folder = 'output'
process_folder(input_folder, output_folder)